# M1 Test Evaluation

M1 pathology-only Cox risk model을 이용하여 TCGA test split과 CPTAC external cohort 성능을 평가한다.

주요 출력:
- case-level prediction table
- C-index
- horizon별 AUROC/AUPRC
- bootstrap confidence interval
- high/low risk log-rank test
- Kaplan-Meier curve


In [ ]:
# 1. Imports and paths
from __future__ import annotations

import json
import math
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy.stats import chi2
from torch import nn
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm.auto import tqdm

try:
    import timm
    from huggingface_hub import hf_hub_download
except ImportError as exc:
    raise ImportError("timm, huggingface_hub가 필요합니다. AGENTS.md의 가상환경에서 실행하세요.") from exc

from scripts.models.discrete_survival import harrell_c_index, horizon_auc_metrics
from scripts.models.m1_pathology_mil import M1ModelConfig, PathologySpatialMIL, sample_tiles

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_PATH = Path("../../data")
PROJECT_DATA_PATH = DATA_PATH / "pancreatic_cancer_pathology"
DST_PATH = PROJECT_DATA_PATH / "dst"
IMAGE_PATH = DST_PATH / "Image"
CLINICAL_PATH = DST_PATH / "Clinical"
MODEL_PATH = Path("../../model") / "pancreatic_cancer_pathology" / "M1" / "UNI2-h"
OUTPUT_PATH = Path("outputs/M1_test")
FIGURE_PATH = OUTPUT_PATH / "figures"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT_PATH = MODEL_PATH / "best_checkpoint.pt"
TRAIN_CONFIG_PATH = MODEL_PATH / "training_config.json"
TCGA_SPLIT_PATH = Path("outputs/M1/m1_tcga_horizon_case_splits.csv")
TCGA_TILE_INDEX_PATH = Path("outputs/M1/m1_tcga_horizon_tile_index.csv")

DEVICE = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("checkpoint:", BEST_CHECKPOINT_PATH, BEST_CHECKPOINT_PATH.exists())
print("config:", TRAIN_CONFIG_PATH, TRAIN_CONFIG_PATH.exists())


In [ ]:
# 2. Label, transform, metrics, and dataset helpers
MONTH_DAYS = 30.4375
HORIZON_MONTHS = [6, 12, 18, 24]
HORIZON_DAYS = np.array([m * MONTH_DAYS for m in HORIZON_MONTHS], dtype=np.float32)
HORIZON_NAMES = [f"dead_by_{m}m" for m in HORIZON_MONTHS]
PATCH_MEAN = (0.485, 0.456, 0.406)
PATCH_STD = (0.229, 0.224, 0.225)


def make_horizon_label_mask(os_time_days: float, os_event: int) -> tuple[np.ndarray, np.ndarray]:
    y = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    mask = np.zeros(len(HORIZON_DAYS), dtype=np.float32)
    if pd.isna(os_time_days) or pd.isna(os_event):
        return y, mask
    os_time_days = float(os_time_days)
    os_event = int(os_event)
    for i, horizon in enumerate(HORIZON_DAYS):
        if os_event == 1 and os_time_days <= float(horizon):
            y[i] = 1.0
            mask[i] = 1.0
        elif os_time_days >= float(horizon):
            y[i] = 0.0
            mask[i] = 1.0
        else:
            y[i] = 0.0
            mask[i] = 0.0
    return y, mask


def load_clinical_label(dataset: str, case_id: str) -> tuple[float | None, int | None]:
    path = CLINICAL_PATH / dataset / f"{case_id}_clinical.json"
    if not path.exists():
        return None, None
    with open(path, "r", encoding="utf-8") as f:
        clinical_json = json.load(f)
    clinical = clinical_json.get("clinical", {})
    return clinical.get("os_time_days"), clinical.get("os_event")


def get_patch_padding(image_size: int, patch_size: int) -> tuple[int, int, int, int]:
    remainder = image_size % patch_size
    if remainder == 0:
        return (0, 0, 0, 0)
    total_pad = patch_size - remainder
    left = total_pad // 2
    right = total_pad - left
    top = total_pad // 2
    bottom = total_pad - top
    return (left, top, right, bottom)


def get_eval_transform(image_size: int, patch_size: int):
    return transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.Pad(get_patch_padding(image_size, patch_size), fill=255),
        transforms.ToTensor(),
        transforms.Normalize(mean=PATCH_MEAN, std=PATCH_STD),
    ])


def parse_xy_from_tile_path(tile_path: str) -> tuple[int, int]:
    match = re.search(r"_x(\d+)_y(\d+)_", Path(tile_path).name)
    if not match:
        return 0, 0
    return int(match.group(1)), int(match.group(2))


def add_coordinate_columns(tile_df: pd.DataFrame, slide_width: float | None = None, slide_height: float | None = None) -> pd.DataFrame:
    df = tile_df.copy()
    if "x" not in df.columns or "y" not in df.columns:
        if {"x_level0", "y_level0"}.issubset(df.columns):
            df["x"] = df["x_level0"].astype(int)
            df["y"] = df["y_level0"].astype(int)
        elif {"x_total_matrix", "y_total_matrix"}.issubset(df.columns):
            df["x"] = df["x_total_matrix"].astype(int)
            df["y"] = df["y_total_matrix"].astype(int)
        else:
            xy = df["tile_path"].map(parse_xy_from_tile_path)
            df["x"] = [v[0] for v in xy]
            df["y"] = [v[1] for v in xy]
    source_size = df["source_tile_size_px"].astype(float) if "source_tile_size_px" in df.columns else 1024.0
    if slide_width is None:
        slide_width = float((df["x"].astype(float) + source_size).max())
    if slide_height is None:
        slide_height = float((df["y"].astype(float) + source_size).max())
    df["slide_width"] = slide_width
    df["slide_height"] = slide_height
    df["x_norm"] = df["x"].astype(float) / slide_width
    df["y_norm"] = df["y"].astype(float) / slide_height
    df["x_center_norm"] = (df["x"].astype(float) + source_size / 2.0) / slide_width
    df["y_center_norm"] = (df["y"].astype(float) + source_size / 2.0) / slide_height
    df["w_norm"] = source_size / slide_width
    df["h_norm"] = source_size / slide_height
    return df


class M1EvalDataset(Dataset):
    def __init__(self, slide_manifest: pd.DataFrame, tile_index: pd.DataFrame):
        self.slide_manifest = slide_manifest.reset_index(drop=True).copy()
        self.tile_groups = {
            case_id: group.sort_values(["y", "x"]).reset_index(drop=True)
            for case_id, group in tile_index.groupby("case_id")
        }

    def __len__(self):
        return len(self.slide_manifest)

    def __getitem__(self, idx):
        row = self.slide_manifest.iloc[idx]
        tiles = self.tile_groups[row["case_id"]]
        coords = tiles[["x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"]].to_numpy(np.float32)
        labels = row[HORIZON_NAMES].to_numpy(np.float32)
        masks = row[[f"mask_{name}" for name in HORIZON_NAMES]].to_numpy(np.float32)
        return {
            "dataset": row["dataset"],
            "case_id": row["case_id"],
            "tile_paths": tiles["tile_path"].tolist(),
            "coords": torch.from_numpy(coords),
            "labels": torch.from_numpy(labels),
            "masks": torch.from_numpy(masks),
            "os_time_days": torch.tensor(float(row["os_time_days"]), dtype=torch.float32),
            "os_event": torch.tensor(int(row["os_event"]), dtype=torch.long),
            "n_tiles_total": len(tiles),
        }


def bootstrap_ci(values_df: pd.DataFrame, metric_fn, n_boot: int = 1000, seed: int = 42) -> tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    point = float(metric_fn(values_df))
    boots = []
    n = len(values_df)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        sample = values_df.iloc[idx]
        try:
            val = float(metric_fn(sample))
        except Exception:
            val = np.nan
        if np.isfinite(val):
            boots.append(val)
    if len(boots) == 0:
        return point, np.nan, np.nan
    return point, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def logrank_test(times: np.ndarray, events: np.ndarray, groups: np.ndarray) -> dict:
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    groups = np.asarray(groups, dtype=int)
    event_times = np.sort(np.unique(times[events == 1]))
    observed = expected = variance = 0.0
    for t in event_times:
        at_risk = times >= t
        events_at_t = (times == t) & (events == 1)
        n = at_risk.sum()
        n1 = (at_risk & (groups == 1)).sum()
        d = events_at_t.sum()
        d1 = (events_at_t & (groups == 1)).sum()
        if n <= 1:
            continue
        observed += d1
        expected += d * n1 / n
        variance += (n1 / n) * (1 - n1 / n) * d * (n - d) / max(n - 1, 1)
    z2 = ((observed - expected) ** 2 / variance) if variance > 0 else np.nan
    p = float(chi2.sf(z2, df=1)) if np.isfinite(z2) else np.nan
    return {"observed_high": observed, "expected_high": expected, "chi2": z2, "p_value": p}


def km_curve(times: np.ndarray, events: np.ndarray) -> pd.DataFrame:
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    rows = [{"time": 0.0, "survival": 1.0, "n_at_risk": len(times), "n_event": 0}]
    survival = 1.0
    for t in np.sort(np.unique(times[events == 1])):
        at_risk = (times >= t).sum()
        n_event = ((times == t) & (events == 1)).sum()
        survival *= 1.0 - n_event / max(at_risk, 1)
        rows.append({"time": float(t), "survival": float(survival), "n_at_risk": int(at_risk), "n_event": int(n_event)})
    return pd.DataFrame(rows)


In [ ]:
# 3. Load trained M1 checkpoint and UNI/UNI2-h feature extractor
with open(TRAIN_CONFIG_PATH, "r", encoding="utf-8") as f:
    train_config = json.load(f)
print(json.dumps(train_config, indent=2, ensure_ascii=False))

PATCH_INPUT_SIZE = int(train_config.get("patch_input_size", 512))
MAX_TILES_PER_SLIDE = int(train_config.get("max_tiles_per_slide", 10_000))
FEATURE_BATCH_SIZE = int(train_config.get("feature_batch_size", 64))
SPATIAL_DIM = int(train_config.get("spatial_dim", 128))
FUSION_DIM = int(train_config.get("fusion_dim", 256))
MIL_HIDDEN_DIM = int(train_config.get("mil_hidden_dim", 128))
DROPOUT = float(train_config.get("dropout", 0.40))
USE_SPATIAL_EMBEDDING = bool(train_config.get("use_spatial_embedding", False))
POOLING_MODE = str(train_config.get("pooling_mode", "mean"))
N_OUTPUTS = len(HORIZON_NAMES)
FEATURE_EXTRACTOR_NAME = str(train_config.get("feature_extractor", "UNI2-h"))

UNI_BACKBONES = {
    "UNI": {
        "repo_id": "MahmoodLab/UNI",
        "filename": "pytorch_model.bin",
        "feature_dim": 1024,
        "timm_kwargs": {
            "model_name": "vit_large_patch16_224",
            "img_size": 224,
            "patch_size": 16,
            "init_values": 1e-5,
            "num_classes": 0,
            "dynamic_img_size": True,
        },
    },
    "UNI2-h": {
        "repo_id": "MahmoodLab/UNI2-h",
        "filename": "pytorch_model.bin",
        "feature_dim": 1536,
        "timm_kwargs": {
            "model_name": "vit_giant_patch14_224",
            "img_size": 224,
            "patch_size": 14,
            "depth": 24,
            "num_heads": 24,
            "init_values": 1e-5,
            "embed_dim": 1536,
            "mlp_ratio": 2.66667 * 2,
            "num_classes": 0,
            "no_embed_class": True,
            "mlp_layer": timm.layers.SwiGLUPacked,
            "act_layer": torch.nn.SiLU,
            "reg_tokens": 8,
            "dynamic_img_size": True,
        },
    },
}


def load_uni_feature_extractor(name: str, device: torch.device, local_files_only: bool = True) -> tuple[nn.Module, int, int]:
    info = UNI_BACKBONES[name]
    kwargs = dict(info["timm_kwargs"])
    model_name = kwargs.pop("model_name")
    model = timm.create_model(model_name, pretrained=False, **kwargs)
    checkpoint_path = hf_hub_download(
        repo_id=info["repo_id"],
        filename=info["filename"],
        local_files_only=local_files_only,
    )
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    if isinstance(state_dict, dict) and "model" in state_dict:
        state_dict = state_dict["model"]
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    if missing or unexpected:
        print("load_state_dict warning", {"missing": missing, "unexpected": unexpected})
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad = False
    return model, int(info["feature_dim"]), int(info["timm_kwargs"]["patch_size"])

feature_extractor, feature_dim, feature_patch_size = load_uni_feature_extractor(FEATURE_EXTRACTOR_NAME, DEVICE, local_files_only=True)
model = PathologySpatialMIL(
    feature_extractor=feature_extractor,
    config=M1ModelConfig(
        feature_dim=feature_dim,
        coord_dim=6,
        spatial_dim=SPATIAL_DIM,
        fusion_dim=FUSION_DIM,
        mil_hidden_dim=MIL_HIDDEN_DIM,
        n_outputs=N_OUTPUTS,
        dropout=DROPOUT,
        max_tiles=MAX_TILES_PER_SLIDE,
        feature_batch_size=FEATURE_BATCH_SIZE,
        freeze_feature_extractor=True,
        use_spatial_embedding=USE_SPATIAL_EMBEDDING,
        pooling_mode=POOLING_MODE,
    ),
).to(DEVICE)

checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location="cpu")
missing, unexpected = model.load_state_dict(checkpoint["model_state_dict"], strict=False)
print("checkpoint epoch:", checkpoint.get("epoch"), "best_monitor_score:", checkpoint.get("best_monitor_score"))
print("missing:", missing)
print("unexpected:", unexpected)
model.eval()

eval_transform = get_eval_transform(PATCH_INPUT_SIZE, feature_patch_size)
print("model ready", {"feature_dim": feature_dim, "patch_size": feature_patch_size, "padding": get_patch_padding(PATCH_INPUT_SIZE, feature_patch_size)})


In [ ]:
# 4. Build TCGA test and CPTAC external evaluation datasets

def build_tcga_eval() -> tuple[pd.DataFrame, pd.DataFrame]:
    split_df = pd.read_csv(TCGA_SPLIT_PATH)
    test_df = split_df[split_df["split"].eq("test")].copy()
    tile_index = pd.read_csv(TCGA_TILE_INDEX_PATH)
    tile_index = tile_index[tile_index["case_id"].isin(test_df["case_id"])].copy()
    if not {"x_norm", "y_norm", "x_center_norm", "y_center_norm", "w_norm", "h_norm"}.issubset(tile_index.columns):
        tile_index = add_coordinate_columns(tile_index)
    return test_df, tile_index


def build_cptac_eval() -> tuple[pd.DataFrame, pd.DataFrame]:
    summary = pd.read_csv(IMAGE_PATH / "CPTAC_PDAC" / "tile_summary.csv")
    summary = summary[summary["status"].eq("done") & summary["metadata_path"].notna()].copy()
    records = []
    tile_tables = []
    for _, row in tqdm(summary.iterrows(), total=len(summary), desc="Load CPTAC tile metadata"):
        case_id = str(row["case_id"])
        metadata_path = Path(row["metadata_path"])
        if not metadata_path.exists():
            continue
        tile_df = pd.read_csv(metadata_path)
        tile_df = tile_df[tile_df["tile_path"].map(lambda p: Path(p).exists())].copy()
        if tile_df.empty:
            continue
        tile_df["dataset"] = "CPTAC_PDAC"
        tile_df["case_id"] = case_id
        tile_df = add_coordinate_columns(tile_df)
        os_time, os_event = load_clinical_label("CPTAC_PDAC", case_id)
        if os_time is None or os_event is None:
            os_time = tile_df["OS_time"].iloc[0] if "OS_time" in tile_df.columns else np.nan
            os_event = tile_df["OS_event"].iloc[0] if "OS_event" in tile_df.columns else np.nan
        y, mask = make_horizon_label_mask(os_time, os_event)
        if pd.isna(os_time) or pd.isna(os_event) or mask.sum() == 0:
            continue
        rec = {
            "dataset": "CPTAC_PDAC",
            "case_id": case_id,
            "os_time_days": float(os_time),
            "os_event": int(os_event),
            "known_horizon_count": int(mask.sum()),
            "n_tiles": len(tile_df),
        }
        for i, name in enumerate(HORIZON_NAMES):
            rec[name] = float(y[i])
            rec[f"mask_{name}"] = float(mask[i])
        records.append(rec)
        tile_tables.append(tile_df)
    slide_df = pd.DataFrame(records)
    tile_index = pd.concat(tile_tables, ignore_index=True) if tile_tables else pd.DataFrame()
    return slide_df, tile_index


tcga_test_df, tcga_test_tiles = build_tcga_eval()
cptac_df, cptac_tiles = build_cptac_eval()

tcga_test_dataset = M1EvalDataset(tcga_test_df, tcga_test_tiles)
cptac_dataset = M1EvalDataset(cptac_df, cptac_tiles)

print("TCGA test cases:", len(tcga_test_dataset), "tiles:", len(tcga_test_tiles))
print("CPTAC external cases:", len(cptac_dataset), "tiles:", len(cptac_tiles))
display(tcga_test_df.head())
display(cptac_df.head())


In [ ]:
# 5. Run inference

PROB_COLS = [f"prob_{name}" for name in HORIZON_NAMES]
LOGIT_COLS = [f"logit_{name}" for name in HORIZON_NAMES]
RISK_SCORE_COL = "prob_dead_by_24m"  # KM/C-index용 대표 위험도


def load_tile_tensor_batch(tile_paths: list[str]) -> torch.Tensor:
    images = []
    for path in tile_paths:
        with Image.open(path) as image_file:
            image = image_file.convert("RGB")
        images.append(eval_transform(image))
    return torch.stack(images, dim=0).to(DEVICE, non_blocking=True)


@torch.no_grad()
def predict_dataset(dataset: M1EvalDataset, dataset_name: str) -> pd.DataFrame:
    rows = []
    for sample in tqdm(dataset, desc=f"Predict {dataset_name}", unit="case"):
        selected_paths, selected_coords, _ = sample_tiles(
            sample["tile_paths"],
            sample["coords"],
            max_tiles=MAX_TILES_PER_SLIDE,
            training=False,
        )
        tile_images = load_tile_tensor_batch(selected_paths)
        coords = selected_coords.to(DEVICE, non_blocking=True)
        outputs = model(tile_images, coords)
        logits = outputs["logits"].detach().cpu().reshape(-1).numpy().astype(float)
        probs = 1.0 / (1.0 + np.exp(-logits))

        row = {
            "dataset": dataset_name,
            "case_id": sample["case_id"],
            "risk_score": float(probs[-1]),
            "os_time_days": float(sample["os_time_days"]),
            "os_event": int(sample["os_event"]),
            "n_tiles_total": int(sample["n_tiles_total"]),
            "n_tiles_used": len(selected_paths),
        }
        labels = sample["labels"].numpy()
        masks = sample["masks"].numpy()
        for i, name in enumerate(HORIZON_NAMES):
            row[name] = float(labels[i])
            row[f"mask_{name}"] = float(masks[i])
            row[f"logit_{name}"] = float(logits[i])
            row[f"prob_{name}"] = float(probs[i])

        tile_risk = outputs.get("tile_risk_score")
        if tile_risk is not None:
            tile_risk_np = tile_risk.detach().cpu().numpy().astype(float)
            row["tile_risk_mean"] = float(tile_risk_np.mean())
            row["tile_risk_std"] = float(tile_risk_np.std())
            row["tile_risk_max"] = float(tile_risk_np.max())
            top10_n = max(1, int(np.ceil(len(tile_risk_np) * 0.10)))
            row["tile_risk_top10_mean"] = float(np.sort(tile_risk_np)[-top10_n:].mean())

        rows.append(row)
        del tile_images, coords, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    pred_df = pd.DataFrame(rows)
    pred_df["risk_percentile"] = pred_df["risk_score"].rank(pct=True)
    return pred_df


tcga_pred_df = predict_dataset(tcga_test_dataset, "TCGA_test")
cptac_pred_df = predict_dataset(cptac_dataset, "CPTAC_external")

tcga_pred_df.to_csv(OUTPUT_PATH / "m1_tcga_test_predictions.csv", index=False)
cptac_pred_df.to_csv(OUTPUT_PATH / "m1_cptac_external_predictions.csv", index=False)

display(tcga_pred_df.head())
display(cptac_pred_df.head())
print("saved:", OUTPUT_PATH)


In [ ]:
# 6. Metrics, bootstrap CI, log-rank test, and KM curves

def get_score_matrix(pred_df: pd.DataFrame) -> np.ndarray:
    return pred_df[PROB_COLS].to_numpy(float)


def evaluate_predictions(pred_df: pd.DataFrame, name: str, n_boot: int = 1000) -> tuple[dict, pd.DataFrame]:
    labels = pred_df[HORIZON_NAMES].to_numpy(float)
    masks = pred_df[[f"mask_{h}" for h in HORIZON_NAMES]].to_numpy(float)
    score_matrix = get_score_matrix(pred_df)
    metrics = {
        "dataset": name,
        "n_cases": len(pred_df),
        "n_events": int(pred_df["os_event"].sum()),
        "c_index": harrell_c_index(pred_df["os_time_days"], pred_df["os_event"], pred_df["risk_score"]),
    }
    metrics.update(horizon_auc_metrics(labels, masks, score_matrix, HORIZON_NAMES))

    c_point, c_low, c_high = bootstrap_ci(
        pred_df,
        lambda d: harrell_c_index(d["os_time_days"], d["os_event"], d["risk_score"]),
        n_boot=n_boot,
        seed=SEED,
    )
    auc_point, auc_low, auc_high = bootstrap_ci(
        pred_df,
        lambda d: horizon_auc_metrics(
            d[HORIZON_NAMES].to_numpy(float),
            d[[f"mask_{h}" for h in HORIZON_NAMES]].to_numpy(float),
            d[PROB_COLS].to_numpy(float),
            HORIZON_NAMES,
        )["mean_auroc"],
        n_boot=n_boot,
        seed=SEED + 1,
    )
    metrics.update({
        "c_index_boot": c_point,
        "c_index_ci_low": c_low,
        "c_index_ci_high": c_high,
        "mean_auroc_boot": auc_point,
        "mean_auroc_ci_low": auc_low,
        "mean_auroc_ci_high": auc_high,
        "risk_score_definition": RISK_SCORE_COL,
    })

    cutoff = float(pred_df["risk_score"].median())
    group = (pred_df["risk_score"] >= cutoff).astype(int).to_numpy()
    lr = logrank_test(pred_df["os_time_days"].to_numpy(float), pred_df["os_event"].to_numpy(int), group)
    metrics.update({"risk_cutoff_median": cutoff, **{f"logrank_{k}": v for k, v in lr.items()}})

    group_df = pred_df.copy()
    group_df["risk_group"] = np.where(group == 1, "High risk", "Low risk")
    return metrics, group_df


metric_rows = []
group_tables = []
for name, pred in [("TCGA_test", tcga_pred_df), ("CPTAC_external", cptac_pred_df)]:
    metrics, group_df = evaluate_predictions(pred, name, n_boot=1000)
    metric_rows.append(metrics)
    group_tables.append(group_df)

metric_df = pd.DataFrame(metric_rows)
group_pred_df = pd.concat(group_tables, ignore_index=True)
metric_df.to_csv(OUTPUT_PATH / "m1_test_metric_summary.csv", index=False)
group_pred_df.to_csv(OUTPUT_PATH / "m1_test_predictions_with_risk_group.csv", index=False)

display(metric_df.T)
print("saved metrics:", OUTPUT_PATH / "m1_test_metric_summary.csv")

for name, group_df in group_pred_df.groupby("dataset"):
    plt.figure(figsize=(6, 5))
    for group_name, part in group_df.groupby("risk_group"):
        km = km_curve(part["os_time_days"].to_numpy(float), part["os_event"].to_numpy(int))
        plt.step(km["time"] / 30.4375, km["survival"], where="post", label=f"{group_name} (n={len(part)})")
    lr_p = metric_df.loc[metric_df["dataset"].eq(name), "logrank_p_value"].iloc[0]
    c_idx = metric_df.loc[metric_df["dataset"].eq(name), "c_index"].iloc[0]
    plt.title(f"{name} KM by 24m predicted risk\nC-index={c_idx:.3f}, log-rank p={lr_p:.3g}")
    plt.xlabel("Months")
    plt.ylabel("Overall survival probability")
    plt.ylim(0, 1.02)
    plt.grid(alpha=0.25)
    plt.legend()
    out = FIGURE_PATH / f"{name}_km_risk_group.png"
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print("saved:", out)


# 7. Publication-ready Tables and Figures

TCGA test와 CPTAC external 성능을 논문용 table/figure 형태로 정리한다.

생성 항목:
- cohort/risk-group summary table
- overall performance table with bootstrap CI
- horizon-wise AUROC/AUPRC table with bootstrap CI
- dataset comparison figure
- horizon-wise ROC/PR curves
- risk score distribution and OS scatter
- combined Kaplan-Meier panel


In [ ]:
# 7. Publication-ready tables and figures
from sklearn.metrics import average_precision_score, auc, precision_recall_curve, roc_auc_score, roc_curve

TABLE_PATH = OUTPUT_PATH / "tables"
TABLE_PATH.mkdir(parents=True, exist_ok=True)

def save_markdown_table(df: pd.DataFrame, path: Path) -> None:
    try:
        df.to_markdown(path, index=False)
    except ImportError:
        print(f"tabulate가 없어 markdown 저장은 건너뜁니다: {path}")


N_BOOT_FIGURE = 1000
ALL_PRED_DF = pd.concat([tcga_pred_df, cptac_pred_df], ignore_index=True)


def format_ci(point: float, low: float, high: float, digits: int = 3) -> str:
    if not np.isfinite(point):
        return "NA"
    if not np.isfinite(low) or not np.isfinite(high):
        return f"{point:.{digits}f}"
    return f"{point:.{digits}f} ({low:.{digits}f}-{high:.{digits}f})"


def horizon_score_col(horizon: str) -> str:
    return f"prob_{horizon}"


def horizon_metric_point(df: pd.DataFrame, horizon: str, metric: str) -> float:
    mask_col = f"mask_{horizon}"
    known = df[mask_col].eq(1)
    y_true = df.loc[known, horizon].astype(int).to_numpy()
    y_score = df.loc[known, horizon_score_col(horizon)].astype(float).to_numpy()
    if len(np.unique(y_true)) < 2:
        return np.nan
    if metric == "auroc":
        return float(roc_auc_score(y_true, y_score))
    if metric == "auprc":
        return float(average_precision_score(y_true, y_score))
    raise ValueError(metric)


def bootstrap_horizon_metric(df: pd.DataFrame, horizon: str, metric: str, n_boot: int = N_BOOT_FIGURE) -> tuple[float, float, float]:
    return bootstrap_ci(df, lambda d: horizon_metric_point(d, horizon, metric), n_boot=n_boot, seed=SEED + hash((horizon, metric)) % 10000)


# Cohort and risk-group summary.
cohort_rows = []
risk_group_rows = []
for dataset_name, df in ALL_PRED_DF.groupby("dataset"):
    cohort_rows.append({
        "dataset": dataset_name,
        "n_cases": len(df),
        "n_events": int(df["os_event"].sum()),
        "event_rate": float(df["os_event"].mean()),
        "median_os_days": float(df["os_time_days"].median()),
        "iqr_os_days": f"{df['os_time_days'].quantile(0.25):.1f}-{df['os_time_days'].quantile(0.75):.1f}",
        "median_risk_score": float(df["risk_score"].median()),
        "iqr_risk_score": f"{df['risk_score'].quantile(0.25):.3f}-{df['risk_score'].quantile(0.75):.3f}",
        "median_tiles_used": float(df["n_tiles_used"].median()),
    })
    cutoff = float(df["risk_score"].median())
    tmp = df.copy()
    tmp["risk_group"] = np.where(tmp["risk_score"] >= cutoff, "High", "Low")
    for risk_group, part in tmp.groupby("risk_group"):
        risk_group_rows.append({
            "dataset": dataset_name,
            "risk_group": risk_group,
            "n_cases": len(part),
            "n_events": int(part["os_event"].sum()),
            "event_rate": float(part["os_event"].mean()),
            "median_os_days": float(part["os_time_days"].median()),
            "median_risk_score": float(part["risk_score"].median()),
        })

cohort_summary_df = pd.DataFrame(cohort_rows)
risk_group_summary_df = pd.DataFrame(risk_group_rows)
cohort_summary_df.to_csv(TABLE_PATH / "cohort_summary.csv", index=False)
risk_group_summary_df.to_csv(TABLE_PATH / "risk_group_summary.csv", index=False)

# Compact overall performance table.
overall_rows = []
for _, row in metric_df.iterrows():
    overall_rows.append({
        "dataset": row["dataset"],
        "n_cases": int(row["n_cases"]),
        "n_events": int(row["n_events"]),
        "C-index (95% CI)": format_ci(row["c_index"], row["c_index_ci_low"], row["c_index_ci_high"]),
        "Mean AUROC (95% CI)": format_ci(row["mean_auroc"], row["mean_auroc_ci_low"], row["mean_auroc_ci_high"]),
        "Mean AUPRC": f"{row['mean_auprc']:.3f}",
        "Log-rank p": f"{row['logrank_p_value']:.3g}",
    })
overall_performance_table = pd.DataFrame(overall_rows)
overall_performance_table.to_csv(TABLE_PATH / "overall_performance_table.csv", index=False)
save_markdown_table(overall_performance_table, TABLE_PATH / "overall_performance_table.md")

# Horizon-wise metric table with bootstrap CI.
horizon_rows = []
for dataset_name, df in ALL_PRED_DF.groupby("dataset"):
    for horizon in HORIZON_NAMES:
        known = df[f"mask_{horizon}"].eq(1)
        y_true = df.loc[known, horizon].astype(int)
        prevalence = float(y_true.mean()) if len(y_true) else np.nan
        auroc, auroc_low, auroc_high = bootstrap_horizon_metric(df, horizon, "auroc")
        auprc, auprc_low, auprc_high = bootstrap_horizon_metric(df, horizon, "auprc")
        horizon_rows.append({
            "dataset": dataset_name,
            "horizon": horizon,
            "known_n": int(known.sum()),
            "positive_n": int(y_true.sum()) if len(y_true) else 0,
            "positive_prevalence": prevalence,
            "AUROC": auroc,
            "AUROC_low": auroc_low,
            "AUROC_high": auroc_high,
            "AUROC (95% CI)": format_ci(auroc, auroc_low, auroc_high),
            "AUPRC": auprc,
            "AUPRC_low": auprc_low,
            "AUPRC_high": auprc_high,
            "AUPRC (95% CI)": format_ci(auprc, auprc_low, auprc_high),
            "random_AUPRC_baseline": prevalence,
        })
horizon_metric_table = pd.DataFrame(horizon_rows)
horizon_metric_table.to_csv(TABLE_PATH / "horizon_metric_table.csv", index=False)
save_markdown_table(horizon_metric_table[["dataset", "horizon", "known_n", "positive_n", "positive_prevalence", "AUROC (95% CI)", "AUPRC (95% CI)", "random_AUPRC_baseline"]], TABLE_PATH / "horizon_metric_table.md")

print("Cohort summary")
display(cohort_summary_df)
print("Risk-group summary")
display(risk_group_summary_df)
print("Overall performance")
display(overall_performance_table)
print("Horizon-wise performance")
display(horizon_metric_table)

# Figure 1: overall performance comparison with CIs.
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
plot_specs = [
    ("c_index", "c_index_ci_low", "c_index_ci_high", "C-index"),
    ("mean_auroc", "mean_auroc_ci_low", "mean_auroc_ci_high", "Mean AUROC"),
    ("mean_auprc", None, None, "Mean AUPRC"),
]
for ax, (metric, low_col, high_col, title) in zip(axes, plot_specs):
    x = np.arange(len(metric_df))
    y = metric_df[metric].astype(float).to_numpy()
    if low_col is not None:
        yerr = np.vstack([y - metric_df[low_col].astype(float).to_numpy(), metric_df[high_col].astype(float).to_numpy() - y])
        ax.bar(x, y, yerr=yerr, capsize=4, color=["#4C78A8", "#F58518"])
    else:
        ax.bar(x, y, color=["#4C78A8", "#F58518"])
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_df["dataset"], rotation=20, ha="right")
    ax.set_title(title)
    ax.set_ylim(0, max(1.0, np.nanmax(y) + 0.15))
    for xi, yi in zip(x, y):
        ax.text(xi, yi + 0.02, f"{yi:.3f}", ha="center", fontsize=9)
plt.tight_layout()
out = FIGURE_PATH / "overall_performance_comparison.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("saved:", out)

# Figure 2: horizon-wise AUROC/AUPRC grouped bars.
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, metric, title in [(axes[0], "AUROC", "Horizon-wise AUROC"), (axes[1], "AUPRC", "Horizon-wise AUPRC")]:
    pivot = horizon_metric_table.pivot(index="horizon", columns="dataset", values=metric).loc[HORIZON_NAMES]
    x = np.arange(len(HORIZON_NAMES))
    width = 0.36
    for j, dataset_name in enumerate(pivot.columns):
        ax.bar(x + (j - 0.5) * width, pivot[dataset_name].to_numpy(), width=width, label=dataset_name)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels([h.replace("dead_by_", "") for h in HORIZON_NAMES])
    ax.set_title(title)
    ax.set_ylim(0, 1.0)
    ax.legend()
plt.tight_layout()
out = FIGURE_PATH / "horizon_metric_bars.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("saved:", out)

# Figure 3: ROC curves per horizon.
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, horizon in zip(axes.ravel(), HORIZON_NAMES):
    for dataset_name, df in ALL_PRED_DF.groupby("dataset"):
        known = df[f"mask_{horizon}"].eq(1)
        y_true = df.loc[known, horizon].astype(int).to_numpy()
        y_score = df.loc[known, horizon_score_col(horizon)].astype(float).to_numpy()
        if len(np.unique(y_true)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{dataset_name} AUC={roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1)
    ax.set_title(horizon)
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.legend(fontsize=8)
plt.tight_layout()
out = FIGURE_PATH / "horizon_roc_curves.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("saved:", out)

# Figure 4: PR curves per horizon.
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for ax, horizon in zip(axes.ravel(), HORIZON_NAMES):
    for dataset_name, df in ALL_PRED_DF.groupby("dataset"):
        known = df[f"mask_{horizon}"].eq(1)
        y_true = df.loc[known, horizon].astype(int).to_numpy()
        y_score = df.loc[known, horizon_score_col(horizon)].astype(float).to_numpy()
        if len(np.unique(y_true)) < 2:
            continue
        precision, recall, _ = precision_recall_curve(y_true, y_score)
        pr_auc = average_precision_score(y_true, y_score)
        ax.plot(recall, precision, label=f"{dataset_name} AP={pr_auc:.3f}")
        ax.axhline(y_true.mean(), linestyle="--", linewidth=1, alpha=0.5)
    ax.set_title(horizon)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=8)
plt.tight_layout()
out = FIGURE_PATH / "horizon_pr_curves.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("saved:", out)

# Figure 5: risk score distributions and relation with OS.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for i, (dataset_name, df) in enumerate(ALL_PRED_DF.groupby("dataset")):
    jitter = np.random.default_rng(SEED + i).normal(0, 0.03, size=len(df))
    x = np.full(len(df), i) + jitter
    colors = np.where(df["os_event"].eq(1), "#D62728", "#1F77B4")
    axes[0].scatter(x, df["risk_score"], c=colors, alpha=0.8, edgecolor="white", linewidth=0.4)
axes[0].set_xticks(range(ALL_PRED_DF["dataset"].nunique()))
axes[0].set_xticklabels(list(ALL_PRED_DF.groupby("dataset").groups.keys()), rotation=20, ha="right")
axes[0].set_ylabel("24m death probability")
axes[0].set_title("24m risk distribution\nred=event, blue=censored")
for dataset_name, df in ALL_PRED_DF.groupby("dataset"):
    axes[1].scatter(df["risk_score"], df["os_time_days"] / 30.4375, alpha=0.75, label=dataset_name)
axes[1].set_xlabel("24m death probability")
axes[1].set_ylabel("OS time (months)")
axes[1].set_title("Risk score vs observed OS")
axes[1].legend()
plt.tight_layout()
out = FIGURE_PATH / "risk_distribution_and_os_scatter.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("saved:", out)

# Figure 6: combined KM panel.
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, (dataset_name, group_df) in zip(axes, group_pred_df.groupby("dataset")):
    for group_name, part in group_df.groupby("risk_group"):
        km = km_curve(part["os_time_days"].to_numpy(float), part["os_event"].to_numpy(int))
        ax.step(km["time"] / 30.4375, km["survival"], where="post", label=f"{group_name} (n={len(part)})")
    lr_p = metric_df.loc[metric_df["dataset"].eq(dataset_name), "logrank_p_value"].iloc[0]
    c_idx = metric_df.loc[metric_df["dataset"].eq(dataset_name), "c_index"].iloc[0]
    ax.set_title(f"{dataset_name}\nC-index={c_idx:.3f}, log-rank p={lr_p:.3g}")
    ax.set_xlabel("Months")
    ax.grid(alpha=0.25)
    ax.legend()
axes[0].set_ylabel("Overall survival probability")
plt.tight_layout()
out = FIGURE_PATH / "combined_km_risk_group.png"
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("saved:", out)

print("saved tables:", TABLE_PATH)
print("saved figures:", FIGURE_PATH)


In [ ]:
# 7. Paper-ready short summary
summary_lines = []
for _, row in metric_df.iterrows():
    summary_lines.append(
        f"{row['dataset']}: n={int(row['n_cases'])}, events={int(row['n_events'])}, "
        f"C-index={row['c_index']:.3f} (95% CI {row['c_index_ci_low']:.3f}-{row['c_index_ci_high']:.3f}), "
        f"mean AUROC={row['mean_auroc']:.3f} (95% CI {row['mean_auroc_ci_low']:.3f}-{row['mean_auroc_ci_high']:.3f}), "
        f"mean AUPRC={row['mean_auprc']:.3f}, log-rank p={row['logrank_p_value']:.3g}."
    )
for line in summary_lines:
    print(line)

with open(OUTPUT_PATH / "m1_test_summary_text.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines) + "\n")
